# Análisis Pre-Partido — Scouting del Rival
Forma reciente de Chaco, scouting del rival, head-to-head y tabla de posiciones.  
Genera gráficos y el informe narrativo de scouting con Claude.

In [1]:
import os, sys

# Recarga automática de módulos src/ al editarlos (sin reiniciar kernel)
%load_ext autoreload
%autoreload 2

if os.path.basename(os.getcwd()) in ('primera_nacional', 'liga_profesional'):
    os.chdir('..')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import config
from src.scraping.rivals_scraper import scrape_rival_recent, rival_already_scraped, scrape_rival_managers
from src.scraping.standings_scraper import fetch_standings, load_standings
from src.analysis.pre_match import (
    chaco_recent_form, chaco_form_summary,
    rival_form, rival_form_summary,
    rival_tactical_profile, rival_key_players, rival_match_keys,
    head_to_head, head_to_head_summary, head_to_head_goalscorers,
    discipline_alert, rival_best_xi, rival_formations_used, rival_discipline_profile,
    rival_manager_timeline, rival_manager_summary,
    rival_home_away_split, rival_suspension_risk,
)
from src.analysis.post_match import load_match_data, xg_analysis, player_performance_table
from src.analysis.season import (
    is_home_match, threat_by_situation, shot_entropy, assists_network,
)
from src.visualization.pre_match import (
    plot_form_comparison, plot_xg_trend, plot_stats_radar, plot_rival_players,
    plot_head_to_head, plot_discipline_alert,
    plot_rival_best_xi, plot_rival_formations, plot_rival_discipline,
    plot_rival_managers, plot_rival_home_away,
)
from src.visualization.season import (
    save_fig, plot_match_stats, plot_shotmap_4grid_match,
    plot_threat_by_situation, plot_shot_entropy, plot_assists_network,
)
from src.visualization.match import plot_player_ratings_match
from src.reports.pre_match import pre_match_report

# ============================================================
# CONFIGURAR ACÁ
SEASON         = '2026'
NEXT_MATCH_KEY = '2026-10-24_vs_ferro_carril_oeste_a'   # match_key del próximo partido (_h = de local, _a = de visitante; verificar en data/temporadas/<season>/fixture.csv)
RIVAL_SLUG     = 'ferro_carril_oeste'               # clave en config.RIVALS
REFRESCAR_TABLA = True                         # bajar tabla + fixture antes de analizar
CHACO_LAST_N   = 5                             # últimos N partidos de Chaco para forma
GENERATE_REPORT = False
# ============================================================

rival_info = config.RIVALS.get(RIVAL_SLUG, {})
RIVAL_NAME = rival_info.get('name', RIVAL_SLUG.replace('_',' ').title())
_rival_clean = NEXT_MATCH_KEY.split('_vs_',1)[-1] if '_vs_' in NEXT_MATCH_KEY else RIVAL_SLUG
OUTPUTS_DIR = config.pre_outputs_dir(_rival_clean, season=SEASON)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f'Próximo partido : {NEXT_MATCH_KEY}')
print(f'Rival           : {RIVAL_NAME} (slug: {RIVAL_SLUG})')
print(f'Outputs         : {OUTPUTS_DIR}')

Próximo partido : 2026-10-24_vs_ferro_carril_oeste_a
Rival           : Club Ferro Carril Oeste (slug: ferro_carril_oeste)
Outputs         : c:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\outputs\temporadas\2026\pre_partido\ferro_carril_oeste_a


## 1. Tabla de posiciones

In [2]:
# Tabla de las dos zonas + fixture recién bajados (scrapear el partido NO los
# actualiza). headless=False: con headless SofaScore devuelve el captcha.
import io, contextlib
from src.scraping.standings_scraper import fetch_standings, fetch_standings_zona_b
from src.scraping.fixture_scraper import fetch_fixture
if REFRESCAR_TABLA:
    try:
        with contextlib.redirect_stdout(io.StringIO()):   # los emoji revientan en cp1252
            fetch_standings(SEASON, headless=False)
            fetch_standings_zona_b(SEASON, headless=False)
            fetch_fixture(season=SEASON, headless=False, verbose=False)
        print('Tabla y fixture actualizados.')
    except Exception as e:
        print('Aviso al refrescar la tabla:', repr(e))
standings = load_standings(SEASON)
if standings is None:
    standings = fetch_standings(SEASON, headless=False)

if standings is not None:
    print(standings[['position','team_name','matches_played','wins','draws','losses','goal_diff','points']].to_string(index=False))

Tabla cargada desde CSV local.
 position                   team_name  matches_played  wins  draws  losses  goal_diff  points
        1     Club Ferro Carril Oeste              34    19      8       7         10      65
        2                  Godoy Cruz              34    17      9       8         16      60
        3             Deportivo Morón              34    17      7      10         15      58
        4            Deportivo Madryn              34    14     12       8         17      54
        5         Club Atletico Colón              34    14     12       8         14      54
        6           Ciudad de Bolivar              34    13     12       9          1      51
        7             Almirante Brown              34    12     12      10          4      48
        8 Estudiantes de Buenos Aires              34    13      9      12          1      48
        9                   Los Andes              34     9     14      11          1      41
       10              Chaco 

## 2. Forma reciente de Chaco

In [3]:
chaco_form  = chaco_recent_form(n=CHACO_LAST_N, season=SEASON)
chaco_summ  = chaco_form_summary(chaco_form)

print(f'Últimos {CHACO_LAST_N} partidos de Chaco:')
print(f"Forma: {chaco_summ.get('forma_str')}")
print(f"GF {chaco_summ.get('goles_favor')} | GC {chaco_summ.get('goles_contra')}")
print(f"xG prom {chaco_summ.get('xg_favor')} | xGA prom {chaco_summ.get('xg_contra')}")
print(chaco_form[['date','opponent','venue','result','chaco_goals','rival_goals','chaco_xg','rival_xg']].to_string(index=False))

Últimos 5 partidos de Chaco:
Forma: WLDWL
GF 7 | GC 7
xG prom 1.73 | xGA prom 1.77
      date               opponent     venue result  chaco_goals  rival_goals  chaco_xg  rival_xg
2026-09-20        Almirante Brown     Local      W            1            0      1.03      0.28
2026-09-26      Ciudad De Bolivar Visitante      L            1            2      1.69      1.50
2026-10-04 Defensores De Belgrano     Local      D            2            2      1.60      1.80
2026-10-11       Deportivo Madryn Visitante      W            2            1      1.90      1.92
2026-10-18 Central Norte De Salta     Local      L            1            2      2.44      3.36


## 3. Scrapear partidos del rival (si faltan)

In [4]:
# Baja TODOS los partidos jugados por el rival en la temporada (no solo los
# últimos N) — necesario para que formaciones/mejor once/disciplina se calculen
# sobre la muestra completa, no sobre 5 partidos. Es incremental: los partidos
# ya descargados se saltean sin llamadas de red, así que correr esto de nuevo
# más adelante en la temporada solo baja los partidos nuevos.
scraped = scrape_rival_recent(RIVAL_SLUG, n_matches=None, season=SEASON, headless=False)
print(f'\nPartidos de {RIVAL_NAME} disponibles: {len(scraped)}')

# DT de cada partido (endpoint aparte /managers) para detectar cambios de
# entrenador. Idempotente: los que ya tienen managers.json se saltean.
scrape_rival_managers(RIVAL_SLUG, season=SEASON, headless=False)


Scrapeando TODOS los partidos de Club Ferro Carril Oeste en la temporada 2026 (id=36841)...
  34 partidos en el feed.

  [1/34] ya está: 2026-02-14_vs_san_telmo_a
  [2/34] ya está: 2026-02-21_vs_san_miguel_h
  [3/34] ya está: 2026-02-28_vs_club_atletico_colon_a
  [4/34] ya está: 2026-03-14_vs_godoy_cruz_a
  [5/34] ya está: 2026-03-21_vs_los_andes_h
  [6/34] ya está: 2026-03-28_vs_atlanta_a
  [7/34] ya está: 2026-04-04_vs_all_boys_a
  [8/34] ya está: 2026-04-12_vs_estudiantes_de_buenos_air_h
  [9/34] ya está: 2026-04-19_vs_ca_mitre_a
  [10/34] ya está: 2026-04-28_vs_almirante_brown_h
  [11/34] ya está: 2026-05-03_vs_ciudad_de_bolivar_a
  [12/34] ya está: 2026-05-11_vs_defensores_de_belgrano_h
  [13/34] ya está: 2026-05-17_vs_deportivo_madryn_a
  [14/34] ya está: 2026-05-24_vs_central_norte_de_salta_h
  [15/34] ya está: 2026-05-30_vs_racing_de_cordoba_a
  [16/34] ya está: 2026-06-07_vs_chaco_for_ever_a
  [17/34] ya está: 2026-06-14_vs_acassuso_h
  [18/34] ya está: 2026-06-21_vs_deportivo

['2026-02-14_vs_san_telmo_a',
 '2026-02-21_vs_san_miguel_h',
 '2026-02-28_vs_club_atletico_colon_a',
 '2026-03-14_vs_godoy_cruz_a',
 '2026-03-21_vs_los_andes_h',
 '2026-03-28_vs_atlanta_a',
 '2026-04-04_vs_all_boys_a',
 '2026-04-12_vs_estudiantes_de_buenos_air_h',
 '2026-04-19_vs_ca_mitre_a',
 '2026-04-28_vs_almirante_brown_h',
 '2026-05-03_vs_ciudad_de_bolivar_a',
 '2026-05-11_vs_defensores_de_belgrano_h',
 '2026-05-17_vs_deportivo_madryn_a',
 '2026-05-24_vs_central_norte_de_salta_h',
 '2026-05-30_vs_racing_de_cordoba_a',
 '2026-06-07_vs_chaco_for_ever_a',
 '2026-06-14_vs_acassuso_h',
 '2026-06-21_vs_deportivo_moron_h',
 '2026-07-05_vs_san_telmo_h',
 '2026-07-11_vs_san_miguel_a',
 '2026-07-17_vs_club_atletico_colon_h',
 '2026-07-26_vs_deportivo_moron_a',
 '2026-08-01_vs_godoy_cruz_h',
 '2026-08-09_vs_los_andes_a',
 '2026-08-15_vs_atlanta_h',
 '2026-08-24_vs_all_boys_h',
 '2026-08-30_vs_estudiantes_de_buenos_air_a',
 '2026-09-04_vs_ca_mitre_h',
 '2026-09-13_vs_almirante_brown_a',
 '202

## 4. Análisis del rival

In [5]:
rform       = rival_form(RIVAL_SLUG, SEASON)
rform_summ  = rival_form_summary(rform)
rprofile    = rival_tactical_profile(RIVAL_SLUG, SEASON)
rplayers    = rival_key_players(RIVAL_SLUG, SEASON)

print(f'Forma reciente de {RIVAL_NAME}:')
print(f"Forma: {rform_summ.get('forma_str')}")
print(f"GF {rform_summ.get('goles_favor')} | GC {rform_summ.get('goles_contra')}")
if not rform.empty:
    print(rform[['date','opponent','venue','result','goals_for','goals_against']].to_string(index=False))

print(f'\nJugadores clave de {RIVAL_NAME}:')
if not rplayers.empty:
    print(rplayers.to_string(index=False))

# Rendimiento local vs visitante del rival. Como el partido es en nuestra
# cancha, lo que importa es su forma de VISITANTE.
split = rival_home_away_split(RIVAL_SLUG, SEASON)
if not split.empty:
    print('Local vs Visitante del rival:')
    print(split.to_string(index=False))
    WE_ARE_HOME = NEXT_MATCH_KEY.endswith('_h')
    p = os.path.join(OUTPUTS_DIR, 'local_visitante_rival.png')
    plot_rival_home_away(split, rival_name=RIVAL_NAME, we_are_home=WE_ARE_HOME, save_path=p)
    print('OK local_visitante_rival.png')


Forma reciente de Club Ferro Carril Oeste:
Forma: WDDLLWDLWLWWWWDWWWWWWLLWWWDDDWWLDW
GF 38 | GC 28
      date                  opponent     venue result  goals_for  goals_against
2026-02-14                 San Telmo Visitante      W          2              0
2026-02-21                San Miguel     Local      D          1              1
2026-02-28       Club Atletico Colon Visitante      D          1              1
2026-03-14                Godoy Cruz Visitante      L          1              2
2026-03-21                 Los Andes     Local      L          0              2
2026-03-28                   Atlanta Visitante      W          1              0
2026-04-04                  All Boys Visitante      D          0              0
2026-04-12 Estudiantes De Buenos Air     Local      L          0              1
2026-04-19                  Ca Mitre Visitante      W          1              0
2026-04-28           Almirante Brown     Local      L          0              1
2026-05-03         Ci

## 5. Mejor once y formaciones del rival

In [6]:
# "Mejor once": el arquero + los 10 de campo con mas titularidades en la
# muestra scrapeada (no es la alineacion oficial confirmada del proximo partido).
best_xi = rival_best_xi(RIVAL_SLUG, SEASON)
if not best_xi.empty:
    print(f'Mejor once de {RIVAL_NAME} (segun titularidades recientes):')
    print(best_xi[['player_name','position','starts','matches_played','avg_rating','goals','assists']]
         .to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'mejor_once_rival.png')
    plot_rival_best_xi(best_xi, rival_name=RIVAL_NAME, save_path=p)
    print('OK mejor_once_rival.png')

# Formaciones usadas en sus ultimos partidos scrapeados.
formations = rival_formations_used(RIVAL_SLUG, SEASON)
if not formations.empty:
    print(f'Formaciones de {RIVAL_NAME}:')
    print(formations.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'formaciones_rival.png')
    plot_rival_formations(formations, rival_name=RIVAL_NAME, save_path=p)
    print('OK formaciones_rival.png')

# Linea de tiempo del DT del rival (cambio de tecnico en la temporada?).
mgr_tl = rival_manager_timeline(RIVAL_SLUG, SEASON)
if not mgr_tl.empty:
    spells = rival_manager_summary(mgr_tl)
    print(f'DT de {RIVAL_NAME} por periodo:')
    for sp in spells:
        print("  " + sp["manager"] + ": " + sp["from_date"] + " a " + sp["to_date"] + " (" + str(sp["matches"]) + " pj)")
    p = os.path.join(OUTPUTS_DIR, 'dt_timeline_rival.png')
    plot_rival_managers(mgr_tl, rival_name=RIVAL_NAME, save_path=p)
    print('OK dt_timeline_rival.png')
else:
    print('Sin datos de DT descargados (correr scrape_rival_managers).')


Mejor once de Club Ferro Carril Oeste (segun titularidades recientes):
  player_name position  starts  matches_played  avg_rating  goals  assists
   F. Monetti        G      32              32        7.03    0.0      0.0
     E. Osuna        D      30              33        7.05    0.0      1.0
      N. Kihm        D      26              30        7.12    1.0      2.0
     G. Canto        D      22              24        7.10    2.0      0.0
     F. Tévez        D      20              22        6.84    0.0      0.0
F. Obradovich        M      30              31        6.96    2.0      1.0
   M. Kabalin        M      27              32        7.02    3.0      2.0
     E. Hoyos        M      26              29        7.14    3.0      5.0
  Á. González        M      25              31        6.80    2.0      7.0
    E. Dening        F      24              30        6.95    7.0      1.0
    M. Acosta        F       8              13        6.88    1.0      0.0
OK mejor_once_rival.png
Forma

## 6. Historial cara a cara (H2H)
Todos los enfrentamientos con datos disponibles (no solo esta temporada) —
matchea por `team_id` de SofaScore, no por el slug del match_key (que a veces
cambia de una temporada a otra).

In [7]:
h2h = head_to_head(RIVAL_SLUG)
h2h_summ = head_to_head_summary(h2h)
chaco_scorers, rival_scorers = head_to_head_goalscorers(h2h)

if not h2h.empty:
    print(f"Historial vs {RIVAL_NAME}: {h2h_summ['partidos']} partidos "
         f"({h2h_summ['victorias']}V-{h2h_summ['empates']}E-{h2h_summ['derrotas']}D) "
         f"| GF {h2h_summ['goles_favor']} - GC {h2h_summ['goles_contra']}")
    print(h2h[['date','season','venue','chaco_goals','rival_goals','result']].to_string(index=False))

    if not chaco_scorers.empty:
        print(f'\nGoleadores de Chaco vs {RIVAL_NAME}:')
        print(chaco_scorers.to_string(index=False))
    if not rival_scorers.empty:
        print(f'\nGoleadores de {RIVAL_NAME} vs Chaco:')
        print(rival_scorers.to_string(index=False))

    _mgr_note = None
    _tl = rival_manager_timeline(RIVAL_SLUG, SEASON)
    if not _tl.empty:
        _sp = rival_manager_summary(_tl)
        _nch = int(_tl['changed'].sum())
        _mgr_note = f"DT actual del rival: {_sp[-1]['manager']}" + (f' ({_nch} cambios en {SEASON})' if _nch else '')
    p = os.path.join(OUTPUTS_DIR, 'cara_a_cara.png')
    plot_head_to_head(h2h, chaco_scorers, rival_scorers, rival_name=RIVAL_NAME, save_path=p,
                      summary=h2h_summ, manager_note=_mgr_note)
    print('✅ cara_a_cara.png')
else:
    print(f'Sin historial previo con datos vs {RIVAL_NAME} (primer cruce registrado).')

Historial vs Club Ferro Carril Oeste: 1 partidos (0V-0E-1D) | GF 1 - GC 2
      date season venue  chaco_goals  rival_goals result
2026-06-07   2026 Local            1            2      L

Goleadores de Chaco vs Club Ferro Carril Oeste:
     player  goals
I. Enriquez      1

Goleadores de Club Ferro Carril Oeste vs Chaco:
  player  goals
G. Canto      1
E. Hoyos      1
✅ cara_a_cara.png


### Detalle del último cruce
Reutiliza el análisis de post-partido (`src/analysis/post_match.py` y sus
gráficos) sobre el `match_key` del cruce más reciente: comparación de
estadísticas, shotmap del partido y ratings — no solo el resultado.

In [8]:
if not h2h.empty:
    last_row = h2h.iloc[-1]
    last_mk, last_season = last_row['match_key'], last_row['season']
    print(f"Último cruce: {last_mk}  ({last_row['venue']}, resultado {last_row['result']})")

    data = load_match_data(last_mk, season=last_season)
    xg = xg_analysis(last_mk, data=data, season=last_season)
    if xg:
        print(xg['verdict'])
        print(f"xG: Chaco {xg['chaco_xg']} - {xg['rival_xg']} {RIVAL_NAME}")

    home_match = is_home_match(last_mk)

    if data.get('stats') is not None:
        home_label = 'Chaco For Ever' if home_match else RIVAL_NAME
        away_label = RIVAL_NAME if home_match else 'Chaco For Ever'
        p = os.path.join(OUTPUTS_DIR, 'ultimo_cruce_stats.png')
        plot_match_stats(data['stats'], home_label=home_label, away_label=away_label,
                         is_chaco_home=home_match, save_path=p)
        print('✅ ultimo_cruce_stats.png')

    if data.get('shotmap') is not None:
        p = os.path.join(OUTPUTS_DIR, 'ultimo_cruce_shotmap.png')
        plot_shotmap_4grid_match(data['shotmap'], is_home=home_match, chaco_label='Chaco For Ever',
                                 rival_label=RIVAL_NAME, match_label=last_mk, save_path=p)
        print('✅ ultimo_cruce_shotmap.png')

    perf = player_performance_table(last_mk, data=data, season=last_season)
    if not perf.empty:
        p = os.path.join(OUTPUTS_DIR, 'ultimo_cruce_ratings.png')
        plot_player_ratings_match(perf, chaco_label='Chaco For Ever', save_path=p)
        print('✅ ultimo_cruce_ratings.png')
        top = perf.dropna(subset=['rating']).sort_values('rating', ascending=False).head(3)
        print('\nMejores de Chaco en ese partido:')
        print(top[['player_name','rating','goals','assists']].to_string(index=False))

    # Goleadores del rival en ESE partido puntual (no todo el historial acumulado)
    if data.get('incidents') is not None:
        inc = data['incidents']
        rival_goals_last = inc[(inc['type'] == 'goal') & (inc['is_home'] != home_match)]
        if not rival_goals_last.empty:
            nombres = rival_goals_last['player'].dropna().astype(str).tolist()
            print(f"\nGoleadores de {RIVAL_NAME} en ese partido: " + ", ".join(nombres))

Último cruce: 2026-06-07_vs_ferro_carril_oeste_h  (Local, resultado L)
Chaco perdió en xG (0.44 vs 0.90)
xG: Chaco 0.44 - 0.9 Club Ferro Carril Oeste
✅ ultimo_cruce_stats.png
✅ ultimo_cruce_shotmap.png
✅ ultimo_cruce_ratings.png

Mejores de Chaco en ese partido:
 player_name  rating  goals  assists
G. Melgarejo     8.1    0.0      1.0
   N. Caprio     7.3    0.0      0.0
 I. Enriquez     7.3    1.0      0.0

Goleadores de Club Ferro Carril Oeste en ese partido: G. Canto, E. Hoyos


## 7. Perfil ofensivo del rival
Reutiliza tal cual las funciones de temporada (`threat_by_situation`,
`shot_entropy`, `assists_network` de `src/analysis/season.py`) pero aplicadas
a los partidos DEL RIVAL (ya con la temporada completa scrapeada): por dónde
suele atacar más y quiénes son sus goleadores/asistidores.

In [9]:
rival_mks = rival_match_keys(RIVAL_SLUG, SEASON)
rival_base = os.path.join(config.rivals_dir(SEASON), RIVAL_SLUG)

rival_threat = threat_by_situation(rival_mks, rival_base)
if not rival_threat.empty:
    print(f'Amenaza por tipo de jugada de {RIVAL_NAME}:')
    print(rival_threat.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'amenaza_rival.png')
    plot_threat_by_situation(rival_threat, season=SEASON,
                             title=f'Amenaza por Tipo de Jugada – {RIVAL_NAME}', save_path=p)
    print('✅ amenaza_rival.png')

rival_entropy = shot_entropy(rival_mks, rival_base)
if rival_entropy['n_shots']:
    print(f"\nEntropía de tiros de {RIVAL_NAME}: {rival_entropy['entropy']} / "
         f"{rival_entropy['max_entropy']} bits ({rival_entropy['pct_of_max']}% del máximo)")
    p = os.path.join(OUTPUTS_DIR, 'entropia_rival.png')
    plot_shot_entropy(rival_entropy, season=SEASON, team_label=RIVAL_NAME, save_path=p)
    print('✅ entropia_rival.png')

rival_an, rival_ae = assists_network(rival_mks, rival_base)
if not rival_an.empty:
    print(f'\nGoleadores/asistidores de {RIVAL_NAME} en la temporada:')
    print(rival_an.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'red_asistencias_rival.png')
    plot_assists_network(rival_an, rival_ae, season=SEASON,
                         title=f'Red de Asistencias – {RIVAL_NAME}', save_path=p)
    print('✅ red_asistencias_rival.png')

Amenaza por tipo de jugada de Club Ferro Carril Oeste:
    situation  shots_for  shots_against  goals_for  goals_against  xg_for  xg_against
Juego abierto        269            164         25             19   30.57       16.19
       Córner         58             35          4              3    6.03        3.72
   Tiro libre         16             15          0              0    0.63        0.76
      Lateral         47             45          2              3    2.05        2.69
Pelota parada         30             18          4              1    3.23        1.84
        Penal          5              3          4              1    3.80        2.28
✅ amenaza_rival.png

Entropía de tiros de Club Ferro Carril Oeste: 1.454 / 4.644 bits (31.3% del máximo)
✅ entropia_rival.png

Goleadores/asistidores de Club Ferro Carril Oeste en la temporada:
       player  goals  assists
    L. Parisi      9        0
    E. Dening      7        1
    F. García      5        0
     E. Hoyos      3        5

## 7b. Cadena de Markov del rival (cómo gestiona el marcador)
Reutiliza `markov_game_state` de la temporada (la misma cadena que usamos para
Chaco) aplicada a los partidos del rival: qué le pasa cuando empata, cuando se
pone en ventaja de un gol y cuando va perdiendo por uno. Genera `markov_rival.png`
+ las lecturas de scouting en texto.

In [10]:
# Cadena de Markov del estado del marcador DEL RIVAL — el mismo modelo que
# usamos para Chaco en la temporada (src/analysis/season.markov_game_state),
# reutilizado sobre los partidos del rival. Lee cómo se mueve el marcador en
# sus partidos: si desde el empate suele golpear primero o recibir, si sostiene
# las ventajas de un gol o se las empatan, y si reacciona o se hunde cuando pierde.
from src.analysis.pre_match import rival_markov, rival_markov_summary
from src.visualization.season import plot_markov_transitions

rmat, rtrans = rival_markov(RIVAL_SLUG, SEASON)
if rmat is not None and not rmat.empty:
    rmk_summ = rival_markov_summary(rmat, rtrans)
    print(f'Cadena de Markov de {RIVAL_NAME} (estado del marcador, perspectiva propia):')
    print(rmat.round(2).to_string())
    print('\nLecturas de scouting:')
    for _k in ('desde_empate', 'ganando_por_1', 'perdiendo_por_1'):
        _d = rmk_summ.get(_k)
        if _d:
            print(f"  [{_k}] (n={_d.get('n')}): {_d['lectura']}")
    p = os.path.join(OUTPUTS_DIR, 'markov_rival.png')
    plot_markov_transitions(rmat, rtrans, season=SEASON,
                            title=f'Cadena de Markov del Marcador - {RIVAL_NAME}', save_path=p)
    print('\n✅ markov_rival.png')
else:
    print(f'Sin partidos scrapeados de {RIVAL_NAME} todavía — corré primero la celda de scraping (sección 3).')

Cadena de Markov de Club Ferro Carril Oeste (estado del marcador, perspectiva propia):
      S--    S-    S0    S+  S++
S--  0.67  0.33  0.00  0.00  0.0
S-   0.62  0.00  0.38  0.00  0.0
S0   0.00  0.26  0.00  0.74  0.0
S+   0.00  0.00  0.60  0.00  0.4
S++  0.00  0.00  0.00  1.00  0.0

Lecturas de scouting:
  [desde_empate] (n=38): Desde el 0-0 suele golpear primero: se pone en ventaja el 74% de las veces.
  [ganando_por_1] (n=15): Cuando se pone 1-0 le cuesta cerrarlo: lo alcanzan el 60% de las veces.
  [perdiendo_por_1] (n=8): Si se pone 0-1 se cae: el 62% de las veces recibe el segundo antes de reaccionar.

✅ markov_rival.png


## 7c. Comparación Chaco vs rival (equipo y jugadores)
Perfil de juego de los dos equipos, promedios por partido: **radar ampliado**,
**comparación ofensiva** y **defensiva**, y el desglose de **remates realizados**
y **remates recibidos**. Cierra con un **radar de jugadores cruzado** (un jugador
de Chaco vs una figura del rival, escalado al percentil de la liga). Todo se guarda
en la carpeta del partido.

In [11]:
# Comparación de EQUIPOS Chaco vs rival (perfil de juego, promedios por partido)
# + radar ampliado + paneles ofensivo/defensivo/remates + radar de jugadores cruzado.
from src.analysis.pre_match import (
    chaco_two_way_profile, rival_two_way_profile, cross_team_player_pool,
)
from src.visualization.pre_match import (
    plot_team_radar_extended, plot_offensive_comparison,
    plot_defensive_comparison, plot_shots_comparison,
)
from src.analysis.players import compare_players
from src.visualization.players import plot_players_radar

# --- Dupla del radar de jugadores (editá si querés otra; deben estar en el pool) ---
JUGADOR_CHACO = None   # ej. 'D. Valdez'   (None = elige el de más goles/90 de Chaco)
JUGADOR_RIVAL = None   # ej. 'S. Bellatti' (None = elige el de más goles/90 del rival)

cprof = chaco_two_way_profile(SEASON)
rprof = rival_two_way_profile(RIVAL_SLUG, SEASON)

if rprof.get('n_matches'):
    plot_team_radar_extended(cprof, rprof, RIVAL_NAME,
                             save_path=os.path.join(OUTPUTS_DIR, 'radar_ampliado.png'))
    plot_offensive_comparison(cprof, rprof, RIVAL_NAME,
                              save_path=os.path.join(OUTPUTS_DIR, 'comparacion_ofensiva.png'))
    plot_defensive_comparison(cprof, rprof, RIVAL_NAME,
                              save_path=os.path.join(OUTPUTS_DIR, 'comparacion_defensiva.png'))
    plot_shots_comparison(cprof, rprof, RIVAL_NAME, side='for',
                          save_path=os.path.join(OUTPUTS_DIR, 'remates_realizados.png'))
    plot_shots_comparison(cprof, rprof, RIVAL_NAME, side='against',
                          save_path=os.path.join(OUTPUTS_DIR, 'remates_recibidos.png'))
    print('✅ radar_ampliado, comparacion_ofensiva, comparacion_defensiva, remates_realizados, remates_recibidos')

    # Lo que los paneles no muestran: npxG/xGOT, cabeza, pelota parada, palos,
    # penales y km (km/sprints solo si hubo partidos con tracking).
    import pandas as pd
    _filas = [('npxG', 'npxg'), ('xGOT', 'xgot'), ('Remates de cabeza', 'headers'),
              ('Remates de pelota parada', 'setpiece_shots'), ('Tiros al palo', 'woodwork'),
              ('Penales', 'penalties'), ('Km recorridos', 'km'), ('Sprints', 'sprints')]
    comp_extra = pd.DataFrame([{'metrica': lbl,
                                'chaco_favor': cprof.get(f'{k}_for'),
                                'rival_favor': rprof.get(f'{k}_for'),
                                'chaco_contra': cprof.get(f'{k}_against'),
                                'rival_contra': rprof.get(f'{k}_against')} for lbl, k in _filas])
    print(f'\nPromedio por partido (Chaco {cprof.get("n_shotmap")} / {RIVAL_NAME} {rprof.get("n_shotmap")} con shotmap):')
    print(comp_extra.to_string(index=False))
    comp_extra.to_csv(os.path.join(OUTPUTS_DIR, 'comparacion_extra_equipos.csv'), index=False)

    # Radar de jugadores cruzado (uno de Chaco vs uno del rival, escala liga)
    pool = cross_team_player_pool(RIVAL_SLUG, SEASON)
    rival_mask = pool['team_name'].str.contains(RIVAL_NAME, case=False, na=False)
    # Jugadores de Chaco disponibles: saca a los que se fueron (config.JUGADORES_FUERA)
    chaco_disp = config.excluir_jugadores_fuera(pool[pool['is_chaco']])
    if JUGADOR_CHACO is None:
        _c = chaco_disp.sort_values('goals_p90', ascending=False)
        JUGADOR_CHACO = _c.iloc[0]['player_name'] if not _c.empty else None
    if JUGADOR_RIVAL is None:
        _r = pool[rival_mask].sort_values('goals_p90', ascending=False)
        JUGADOR_RIVAL = _r.iloc[0]['player_name'] if not _r.empty else None
    print(f'Radar jugadores: {JUGADOR_CHACO} (Chaco) vs {JUGADOR_RIVAL} ({RIVAL_NAME})')
    cmp = compare_players(JUGADOR_CHACO, JUGADOR_RIVAL, pool_df=pool)
    if cmp:
        plot_players_radar(cmp, save_path=os.path.join(OUTPUTS_DIR, 'radar_jugadores_cruzado.png'))
        print('✅ radar_jugadores_cruzado.png')
    else:
        print('No encontré alguno de los dos jugadores en el pool (revisá los nombres abajo).')
    print('\nJugadores disponibles para la dupla (por minutos):')
    print('  Chaco :', ', '.join(chaco_disp.sort_values('minutes', ascending=False)['player_name'].head(12)))
    print(f'  {RIVAL_NAME}:', ', '.join(pool[rival_mask].sort_values('minutes', ascending=False)['player_name'].head(12)))
else:
    print(f'Sin partidos scrapeados de {RIVAL_NAME} todavía — corré primero la celda de scraping (sección 3).')

✅ radar_ampliado, comparacion_ofensiva, comparacion_defensiva, remates_realizados, remates_recibidos
Radar jugadores: M. S. Romero (Chaco) vs F. García (Club Ferro Carril Oeste)
✅ radar_jugadores_cruzado.png

Jugadores disponibles para la dupla (por minutos):
  Chaco : A. Luque, D. Valdez, B. Nievas, N. Caprio, J. Carrizo, R. Garay, A. Ojeda, G. Pacheco, M. Silvera, L. Fernandez, A. Lioi, M. S. Romero
  Club Ferro Carril Oeste: F. Monetti, E. Osuna, F. Obradovich, N. Kihm, M. Kabalin, Á. González, G. Canto, E. Hoyos, E. Dening, L. Parisi, F. Tévez, S. Corda


## 7d. Scouting objetivo por puesto: mejores/peores de Mitre vs plantel de Chaco
Usa el pool de scouting de toda la categoría (`src/analysis/scouting.py`, Zona A + Zona B,
temporada completa): métricas por-90 ajustadas por posesión (PAdj), suavizadas por tamaño
de muestra, y percentiladas **dentro de cada rol** (no tiene sentido comparar un central
contra toda la liga). El `score_rol` es el índice compuesto de la sección 3 de la
especificación de scouting; el arquero queda sin score (SofaScore no publica
`goals_prevented`) y se ordena por rating.

Para cada jugador se listan sus 4 mejores estadísticas (la métrica y el percentil dentro
de su puesto), junto con minutos jugados y proporción de titularidad — un score alto con
pocos minutos se marca como `[muestra chica]` y hay que leerlo con cautela.

In [12]:
import pandas as pd
from src.analysis import scouting as SC
from src.analysis import similitud as SIM

MIN_MINUTOS_SCOUT = 300   # más bajo que el default (600): queremos ver a todo el plantel rival, no solo a los consolidados
ORDEN_ROL = ['ARQ', 'DFC', 'LAT', 'MCD', 'MC', 'EXT_MP', 'DC']

pool_scout = SC.pool_liga(SEASON, min_minutos=MIN_MINUTOS_SCOUT, verbose=False)
pool_scout = SC.suavizar_p90(pool_scout)
pool_scout = SC.scores_compuestos(pool_scout)

rival_id = config.RIVALS_2026.get(RIVAL_SLUG, config.RIVALS.get(RIVAL_SLUG, {})).get('id')
rival_pool = pool_scout[pool_scout['team_id'] == rival_id].copy()
# Saca del plantel propio a los jugadores que ya se fueron (config.JUGADORES_FUERA)
chaco_pool = config.excluir_jugadores_fuera(
    pool_scout[pool_scout['team_id'] == config.CHACO_TEAM_ID].copy(), col='nombre')


def _top_stats_jugador(row, n=4):
    """Las n métricas donde mejor percentil saca el jugador dentro de su rol."""
    rol = row['rol']
    if rol not in SC.PESOS_SCORE:
        return []
    filas = [(m, row.get(f'{m}_pct')) for m in SC.metricas_del_score(rol)]
    filas = [(m, p) for m, p in filas if pd.notna(p)]
    filas.sort(key=lambda t: t[1], reverse=True)
    return filas[:n]


def _tabla_equipo(df_equipo, titulo):
    print(f'\n{"=" * 72}\n{titulo}\n{"=" * 72}')
    for rol in ORDEN_ROL:
        sub = df_equipo[df_equipo['rol'] == rol].copy()
        if sub.empty:
            continue
        sub = sub.sort_values('rating' if rol == 'ARQ' else 'score_rol', ascending=False)
        print(f'\n-- {rol} ({SC.ROL_NOMBRE[rol]}) --')
        for idx, (_, r) in enumerate(sub.iterrows()):
            tit = f'{int(r.titular_pj)}/{int(r.pj)} PJ tit.'
            muestra = ' [muestra chica]' if r.get('muestra_chica') else ''
            if rol == 'ARQ':
                resumen = f'rating {r.rating:.2f}'
            else:
                resumen = f'score {r.score_rol:.1f} (percentil {r.score_pct:.0f})'
            tag = ''
            if len(sub) > 1:
                tag = ' <- MEJOR' if idx == 0 else (' <- PEOR' if idx == len(sub) - 1 else '')
            print(f'   {r.nombre:22s} {r.minutos:5.0f} min  {tit:12s} {resumen}{muestra}{tag}')
            for m, p in _top_stats_jugador(r, n=4):
                print(f'       - {SIM._etiqueta(m):22s} percentil {p:.0f}')
    return df_equipo


_tabla_equipo(rival_pool, f'{RIVAL_NAME} (rival) — jugadores por puesto')
_tabla_equipo(chaco_pool, 'Chaco For Ever — jugadores por puesto')

# CSV combinado para el informe / consulta posterior
_out_cols = ['player_id', 'nombre', 'equipo', 'rol', 'banda', 'edad', 'pj', 'titular_pj',
             'minutos', 'pct_minutos_equipo', 'rating', 'rating_sd', 'score_rol', 'score_pct',
             'muestra_chica', 'valor_mercado_eur',
             # del JSON crudo y el shotmap (fuera del score_rol)
             'npxg', 'xgot', 'goles_sin_penal', 'remates_cabeza', 'remates_pelota_parada',
             'tiros_al_palo', 'dribbles_attempted_p90', 'regateado_p90',
             'regates_frenados_pct', 'malos_controles_p90', 'penales_provocados',
             'penales_cometidos', 'penales_errados', 'despejes_en_linea',
             'atajadas_dentro_area', 'km_recorridos_p90', 'sprints_p90', 'pj_tracking']
scouting_puestos = pd.concat([
    rival_pool.assign(lado='rival')[_out_cols + ['lado']],
    chaco_pool.assign(lado='chaco')[_out_cols + ['lado']],
], ignore_index=True)
# Las estadísticas nuevas que no entran en el score, jugador por jugador.
COLS_EXTRA = ['nombre', 'rol', 'minutos', 'npxg', 'xgot', 'goles_sin_penal',
               'remates_cabeza', 'remates_pelota_parada', 'tiros_al_palo',
               'dribbles_attempted_p90', 'regateado_p90', 'regates_frenados_pct',
               'malos_controles_p90', 'penales_provocados', 'penales_cometidos',
               'penales_errados', 'despejes_en_linea', 'atajadas_dentro_area',
               'km_recorridos_p90', 'sprints_p90', 'pj_tracking']
print(f'\nEstadísticas extra de {RIVAL_NAME}:')
display(rival_pool.sort_values('minutos', ascending=False)[COLS_EXTRA].round(2))
print('Estadísticas extra de Chaco:')
display(chaco_pool.sort_values('minutos', ascending=False)[COLS_EXTRA].round(2))

_p = os.path.join(OUTPUTS_DIR, 'scouting_puestos.csv')
scouting_puestos.to_csv(_p, index=False)
print(f'\n✅ scouting_puestos.csv guardado en {_p}')


Club Ferro Carril Oeste (rival) — jugadores por puesto

-- ARQ (Arquero) --
   F. Monetti              2790 min  31/31 PJ tit. rating 7.03

-- DFC (Defensor central) --
   M. Tarón                1026 min  11/16 PJ tit. score 70.4 (percentil 90) <- MEJOR
       - Duelo suelo %          percentil 88
       - Duelo aéreo %          percentil 77
       - Despejes               percentil 76
       - Intercepciones         percentil 76
   G. Canto                1868 min  21/23 PJ tit. score 59.9 (percentil 70)
       - Duelo aéreo %          percentil 93
       - Rating                 percentil 83
       - Pase campo rival %     percentil 76
       - Despejes               percentil 68
   L. P. Biafore            616 min  6/8 PJ tit.  score 49.2 (percentil 47) [muestra chica]
       - Despejes               percentil 99
       - Intercepciones         percentil 88
       - Rating                 percentil 65
       - Pase campo rival %     percentil 60
   F. Tévez                1633 min

## 8. Disciplina — alerta propia y perfil del rival
Jugadores de Chaco a 1 amarilla de la suspensión (cuenta solo los partidos
jugados ANTES de `NEXT_MATCH_KEY`) + tendencia disciplinaria reciente del
rival (NO es una alerta de suspensión de ellos — no tenemos su acumulado
real de temporada, solo los últimos partidos scrapeados).

In [13]:
alert_df = discipline_alert(NEXT_MATCH_KEY, season=SEASON)
# No alertar por jugadores que ya se fueron (config.JUGADORES_FUERA)
alert_df = config.excluir_jugadores_fuera(alert_df, 'player')

if not alert_df.empty:
    print(f'⚠️  Jugadores en riesgo de suspensión para {NEXT_MATCH_KEY}:')
    print(alert_df.to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'alerta_amonestados.png')
    plot_discipline_alert(alert_df, rival_name=RIVAL_NAME, save_path=p)
    print('✅ alerta_amonestados.png')
else:
    print('Sin jugadores en riesgo de suspensión para este partido.')

# Perfil disciplinario del rival. Como bajamos TODA su temporada, el acumulado
# es real: se puede marcar quien esta a una amarilla de la suspension.
rival_team_disc, rival_player_disc = rival_discipline_profile(RIVAL_SLUG, SEASON)
rival_susp = rival_suspension_risk(RIVAL_SLUG, SEASON)
if not rival_susp.empty:
    print('Rival a 1 amarilla de la suspension:')
    print(rival_susp.to_string(index=False))
if not rival_team_disc.empty:
    print(f'\nPerfil disciplinario reciente de {RIVAL_NAME}:')
    print(rival_team_disc.to_string(index=False))
    if not rival_player_disc.empty:
        print(f'\nMás amonestados de {RIVAL_NAME}:')
        print(rival_player_disc.head(6).to_string(index=False))
    p = os.path.join(OUTPUTS_DIR, 'disciplina_rival.png')
    plot_rival_discipline(rival_team_disc, rival_player_disc, rival_name=RIVAL_NAME, save_path=p,
                          suspension_df=rival_susp, full_season=True)
    print('✅ disciplina_rival.png')

⚠️  Jugadores en riesgo de suspensión para 2026-10-24_vs_ferro_carril_oeste_a:
   player  amarillas  rojas  partidos
D. Valdez        9.0    1.0        32
B. Nievas        9.0    0.0        33
G. Canuto        4.0    1.0        31
B. Guerra        4.0    0.0        10
✅ alerta_amonestados.png
Rival a 1 amarilla de la suspension:
player_name position  yellow_cards  to_ban
   E. Osuna        M             9       1
   E. Hoyos        M             4       1

Perfil disciplinario reciente de Club Ferro Carril Oeste:
                                match_key       date                  opponent  fouls  yellow_cards  red_cards
                2026-02-14_vs_san_telmo_a 2026-02-14                 San Telmo     18             3          0
               2026-02-21_vs_san_miguel_h 2026-02-21                San Miguel     11             1          0
      2026-02-28_vs_club_atletico_colon_a 2026-02-28       Club Atletico Colon      9             4          0
               2026-03-14_vs_godoy_cr

## 9. Visualizaciones

In [14]:
import matplotlib
matplotlib.use('Agg')

if not chaco_form.empty and not rform.empty:
    fig = plot_form_comparison(chaco_form, rform, rival_name=RIVAL_NAME)
    save_fig(fig, os.path.join(OUTPUTS_DIR, 'forma_comparada.png'))
    print('✅ forma_comparada.png')

    fig = plot_xg_trend(chaco_form, rform, rival_name=RIVAL_NAME)
    save_fig(fig, os.path.join(OUTPUTS_DIR, 'tendencia_xg.png'))
    print('✅ tendencia_xg.png')

if rprofile:
    # Calcular perfil promedio de Chaco desde sus últimos partidos
    from src.analysis.season import KEY_METRICS, build_season_table, load_matches
    import pandas as pd
    base_dir = config.season_data_dir(SEASON)
    chaco_profile = {}
    if not chaco_form.empty:
        for mk in chaco_form['match_key']:
            sp = os.path.join(base_dir, mk, 'statistics_clean.csv')
            if os.path.exists(sp):
                df_s = pd.read_csv(sp)
                from src.analysis.season import is_home_match
                col = 'home_val' if is_home_match(mk) else 'away_val'
                for _, r in df_s.iterrows():
                    m = r.get('metric','')
                    try:
                        v = float(r.get(col, 0) or 0)
                        chaco_profile[m] = chaco_profile.get(m, []) + [v]
                    except (ValueError, TypeError):
                        pass
        chaco_profile = {k: round(sum(v)/len(v),2) for k,v in chaco_profile.items() if v}

    if chaco_profile and rprofile:
        fig = plot_stats_radar(chaco_profile, rprofile, rival_name=RIVAL_NAME)
        save_fig(fig, os.path.join(OUTPUTS_DIR, 'radar_stats.png'))
        print('✅ radar_stats.png')

if not rplayers.empty:
    fig = plot_rival_players(rplayers, rival_name=RIVAL_NAME)
    save_fig(fig, os.path.join(OUTPUTS_DIR, 'jugadores_rival.png'))
    print('✅ jugadores_rival.png')

✅ forma_comparada.png
✅ tendencia_xg.png
✅ radar_stats.png
✅ jugadores_rival.png


## 10. Generar informe de scouting (Claude)

In [15]:
if GENERATE_REPORT:
    report = pre_match_report(
        next_match_key=NEXT_MATCH_KEY,
        rival_slug=RIVAL_SLUG,
        chaco_form_summary=chaco_summ,
        rival_form_summary=rform_summ,
        rival_profile=rprofile,
        rival_players_df=rplayers if not rplayers.empty else None,
        h2h_df=h2h if not h2h.empty else None,
        standings_df=standings,
        chaco_scorers_df=chaco_scorers if not chaco_scorers.empty else None,
        rival_scorers_df=rival_scorers if not rival_scorers.empty else None,
        discipline_alert_df=alert_df if not alert_df.empty else None,
    )
    print('\n' + report[:600] + '...')
else:
    print('GENERATE_REPORT=False — informe omitido.')

GENERATE_REPORT=False — informe omitido.
